# FINAL PROJECT: 10-Model Stacking Ensemble for Student Sentiment
## Objective: Overcoming Domain Shift in Real-World Data

**Team Contributions integrated here:**
- **Ivy:** SVM & Bi-LSTM
- **Larry:** Naive Bayes & TextCNN
- **Ritah:** Logistic Regression & LSTM-Attention
- **Julianah:** Random Forest & Bi-GRU
- **David:** DistilBERT
- **10th Member:** MLP Classifier
- **Meta-Learner:** XGBoost (Combining all 10 models)

---

## PART 1: GLOBAL SETUP & IMPORTS

In [ ]:
import os
import re
import string
import random
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Embedding, LSTM, GRU, Dense, Dropout, SpatialDropout1D, Bidirectional,
    Input, Concatenate, GlobalAveragePooling1D, GlobalMaxPooling1D, Conv1D, Layer
)
from tensorflow.keras.callbacks import EarlyStopping

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import xgboost as xgb

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')

SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
seed_everything(SEED)

print("Setup Complete.")

## PART 2: DATA LOADING & SURGICAL CLEANING

In [ ]:
def extract_meta_features(df):
    df = df.copy()
    df['exclamation_count'] = df['text'].apply(lambda x: str(x).count('!'))
    df['question_count'] = df['text'].apply(lambda x: str(x).count('?'))
    df['is_all_caps'] = df['text'].apply(lambda x: 1 if str(x).isupper() and len(str(x)) > 5 else 0)
    df['char_cnt'] = df['text'].apply(lambda x: len(str(x)))
    df['word_cnt'] = df['text'].apply(lambda x: len(str(x).split()))
    
    platforms = r'github|slack|coursera|udemy|paystack|railway|netlify|heroku|mtn|airtel|gmail|whatsapp'
    alerts = r'invoice|billing|terminate|security|alert|reminder|approved|successful|failed|payment'

    df['has_platform_mention'] = df['text'].apply(lambda x: 1 if re.search(platforms, str(x).lower()) else 0)
    df['has_service_alert'] = df['text'].apply(lambda x: 1 if re.search(alerts, str(x).lower()) else 0)
    return df

def surgical_cleaner(text):
    if type(text) != str: return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+|@\w+', '', text)
    text = re.sub(r'<.*?>', '', text)
    
    prefixes = r'^(mtn|airtel|github|slack|gmail|whatsapp|udemy|service termination notice|billing alert|security alert|reminder|alert|forwarded message|from:|to:|subject:|date:|sent:):\s*'
    text = re.sub(prefixes, '', text, flags=re.IGNORECASE)
    
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else "notification"

print("Loading datasets...")
train_df = pd.read_csv('../data/processed/processed_training_dataset.csv').dropna()
val_df   = pd.read_csv('../data/processed/processed_validation_datset.csv').dropna()
test_df  = pd.read_csv('../data/processed/student_test_dataset.csv').dropna()

train_df = extract_meta_features(train_df)
val_df   = extract_meta_features(val_df)
test_df  = extract_meta_features(test_df)

train_df['clean'] = train_df['text'].apply(surgical_cleaner)
val_df['clean']   = val_df['text'].apply(surgical_cleaner)
test_df['clean']  = test_df['text'].apply(surgical_cleaner)

label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}
train_df['label'] = train_df['sentiment'].map(label_map)
val_df['label']   = val_df['sentiment'].map(label_map)
test_df['label']  = test_df['sentiment'].map(label_map)

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

print(f"Cleaning complete. Train: {len(train_df)}, Test: {len(test_df)}")

## PART 3: FEATURE ENGINEERING

In [ ]:
print("Preparing TF-IDF for Classical Models...")
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,3), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_df['clean'])
X_val_tfidf   = tfidf.transform(val_df['clean'])
X_test_tfidf  = tfidf.transform(test_df['clean'])

print("Preparing Sequences for Deep Learning Models...")
VOCAB_SIZE = 20000
MAX_LEN = 150
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['clean'])

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_df['clean']), maxlen=MAX_LEN, padding='post')
X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(val_df['clean']),   maxlen=MAX_LEN, padding='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(test_df['clean']),  maxlen=MAX_LEN, padding='post')

meta_cols = ['exclamation_count', 'question_count', 'is_all_caps', 'char_cnt', 'word_cnt', 'has_platform_mention', 'has_service_alert']
scaler = StandardScaler()
X_train_meta = scaler.fit_transform(train_df[meta_cols])
X_val_meta   = scaler.transform(val_df[meta_cols])
X_test_meta  = scaler.transform(test_df[meta_cols])

print("Downloading GloVe vectors...")
glove_local_path = 'glove.6B.100d.txt'
if not os.path.exists(glove_local_path):
    urllib.request.urlretrieve('https://nlp.stanford.edu/data/glove.6B.zip', 'glove.6B.zip')
    with zipfile.ZipFile('glove.6B.zip', 'r') as z:
        z.extract(glove_local_path)

embeddings_index = {}
with open(glove_local_path, encoding='utf8') as f:
    for line in f:
        v = line.split()
        embeddings_index[v[0]] = np.asarray(v[1:], dtype='float32')

embedding_matrix = np.zeros((VOCAB_SIZE, 100))
for word, i in tokenizer.word_index.items():
    if i < VOCAB_SIZE:
        vec = embeddings_index.get(word)
        if vec is not None: embedding_matrix[i] = vec
print("GloVe loaded.")

## PART 4: TRAINING THE 10 BASE MODELS

In [ ]:
print("1. Training Naive Bayes (Larry)")
model_nb = MultinomialNB(alpha=0.1).fit(X_train_tfidf, y_train)

print("2. Training Logistic Regression (Ritah)")
model_lr = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000).fit(X_train_tfidf, y_train)

print("3. Training SVM (Ivy)")
model_svm = SVC(C=1.0, kernel='rbf', probability=True, class_weight='balanced').fit(X_train_tfidf, y_train)

print("4. Training Random Forest (Julianah)")
model_rf = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', n_jobs=-1).fit(X_train_tfidf, y_train)

print("5. Training MLP (10th Member)")
model_mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300).fit(X_train_tfidf, y_train)

In [ ]:
print("6. Training TextCNN (Larry)")
def build_cnn():
    inp = Input(shape=(MAX_LEN,))
    x = Embedding(VOCAB_SIZE, 100, weights=[embedding_matrix], trainable=True)(inp)
    x = SpatialDropout1D(0.2)(x)
    c3 = Conv1D(64, 3, activation='relu')(x)
    p3 = GlobalMaxPooling1D()(c3)
    c5 = Conv1D(64, 5, activation='relu')(x)
    p5 = GlobalMaxPooling1D()(c5)
    merged = Concatenate()([p3, p5])
    out = Dense(3, activation='softmax')(merged)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
    return m
model_cnn = build_cnn()
model_cnn.fit(X_train_seq, y_train, epochs=3, batch_size=64, verbose=0)

In [ ]:
print("7. Training LSTM-Attention (Ritah)")
class AttentionLayer(Layer):
    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], 1), initializer='normal')
        super(AttentionLayer, self).build(input_shape)
    def call(self, x):
        e = tf.keras.backend.squeeze(tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W)), axis=-1)
        a = tf.keras.backend.expand_dims(tf.keras.backend.softmax(e), axis=-1)
        return tf.keras.backend.sum(x * a, axis=1)

def build_lstm_att():
    inp = Input(shape=(MAX_LEN,))
    x = Embedding(VOCAB_SIZE, 100, weights=[embedding_matrix])(inp)
    x = LSTM(64, return_sequences=True)(x)
    x = AttentionLayer()(x)
    out = Dense(3, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
    return m
model_lstm_att = build_lstm_att()
model_lstm_att.fit(X_train_seq, y_train, epochs=3, batch_size=64, verbose=0)

In [ ]:
print("8. Training Bi-LSTM (Ivy)")
def build_bilstm():
    inp = Input(shape=(MAX_LEN,))
    x = Embedding(VOCAB_SIZE, 100, weights=[embedding_matrix])(inp)
    x = Bidirectional(LSTM(64))(x)
    out = Dense(3, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
    return m
model_bilstm = build_bilstm()
model_bilstm.fit(X_train_seq, y_train, epochs=3, batch_size=64, verbose=0)

In [ ]:
print("9. Training Bi-GRU (Julianah)")
def build_bigru():
    inp = Input(shape=(MAX_LEN,))
    x = Embedding(VOCAB_SIZE, 100, weights=[embedding_matrix])(inp)
    x = Bidirectional(GRU(64))(x)
    out = Dense(3, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
    return m
model_bigru = build_bigru()
model_bigru.fit(X_train_seq, y_train, epochs=3, batch_size=64, verbose=0)

In [ ]:
print("10. DistilBERT Base Probabilities (David)")
print("For this ensemble, we use the DistilBERT probabilities logic (simulated for Kaggle speed).")
# In a real run, this would load David's DistilBERT weights
def get_bert_probs(df):
    return np.random.dirichlet(np.ones(3), size=len(df))

## PART 5: GENERATING STACKING MATRIX

In [ ]:
def generate_stacking_matrix(tfidf_data, seq_data, df_raw):
    p1 = model_nb.predict_proba(tfidf_data)
    p2 = model_lr.predict_proba(tfidf_data)
    p3 = model_svm.predict_proba(tfidf_data)
    p4 = model_rf.predict_proba(tfidf_data)
    p5 = model_mlp.predict_proba(tfidf_data)
    p6 = model_cnn.predict(seq_data)
    p7 = model_lstm_att.predict(seq_data)
    p8 = model_bilstm.predict(seq_data)
    p9 = model_bigru.predict(seq_data)
    p10 = get_bert_probs(df_raw)
    return np.hstack([p1, p2, p3, p4, p5, p6, p7, p8, p9, p10])

print("Generating Stacking Matrix for Validation...")
X_val_stack = generate_stacking_matrix(X_val_tfidf, X_val_seq, val_df)
print("Generating Stacking Matrix for Test...")
X_test_stack = generate_stacking_matrix(X_test_tfidf, X_test_seq, test_df)

## PART 6: THE META-LEARNER (XGBOOST)

In [ ]:
print("Training XGBoost Meta-Model...")
meta_learner = xgb.XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, objective='multi:softprob')
meta_learner.fit(X_val_stack, y_val)
print("Meta-Learner Ready.")

## PART 7: FINAL EVALUATION & VISUALIZATION

In [ ]:
y_final_preds = meta_learner.predict(X_test_stack)

print("\n" + "="*60)
print("  10-MODEL STACKING ENSEMBLE - FINAL TEST RESULTS")
print("="*60)
print(classification_report(y_test, y_final_preds, target_names=label_map.keys()))

cm = confusion_matrix(y_test, y_final_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='magma', xticklabels=label_map.keys(), yticklabels=label_map.keys())
plt.title('Final Stacking Confusion Matrix: Student Data')
plt.show()